# ⚡ FastAPI for LLM APIs

**Build production-ready LLM APIs with FastAPI**

---

## 📋 Overview

**What you'll learn:**
- FastAPI basics
- Building LLM endpoints
- Request/response models
- Async operations
- API documentation

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
# Installation
# !pip install fastapi uvicorn pydantic python-multipart

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why FastAPI?

### FastAPI vs Flask:

**Flask:**
```python
from flask import Flask, request

@app.route('/api/chat', methods=['POST'])
def chat():
    data = request.get_json()
    # Manual validation
    # No type hints
    # No auto docs
    return {'response': '...'}
```

**FastAPI:**
```python
from fastapi import FastAPI
from pydantic import BaseModel

class ChatRequest(BaseModel):
    message: str

@app.post('/api/chat')
async def chat(request: ChatRequest):
    # Auto validation ✅
    # Type hints ✅
    # Auto docs ✅
    return {'response': '...'}
```

### Benefits:

- ⚡ **Fast**: Async by default
- 🔍 **Auto validation**: Pydantic models
- 📚 **Auto docs**: Swagger UI included
- 🎯 **Type hints**: Python 3.6+ types
- 🚀 **Production-ready**: Built for scale

## 🏗️ Basic LLM API

In [ ]:
# Save this as main.py and run: uvicorn main:app --reload

print("""
# main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from openai import OpenAI
import os

app = FastAPI(title="LLM API", version="1.0.0")
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Request model
class ChatRequest(BaseModel):
    message: str = Field(..., description="User message", min_length=1, max_length=1000)
    model: str = Field(default="gpt-3.5-turbo", description="Model to use")
    temperature: float = Field(default=0.7, ge=0, le=2)

# Response model
class ChatResponse(BaseModel):
    response: str
    model: str
    tokens: int

@app.post("/api/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    \"\"\"Chat with LLM.\"\"\" 
    
    try:
        response = client.chat.completions.create(
            model=request.model,
            messages=[{"role": "user", "content": request.message}],
            temperature=request.temperature
        )
        
        return ChatResponse(
            response=response.choices[0].message.content,
            model=request.model,
            tokens=response.usage.total_tokens
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "LLM API is running"}

@app.get("/health")
async def health():
    return {"status": "healthy"}

# Run with: uvicorn main:app --reload
# Docs at: http://localhost:8000/docs
""")

## 📝 Request/Response Models

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import List, Optional
from enum import Enum

# Enum for model selection
class ModelType(str, Enum):
    GPT35 = "gpt-3.5-turbo"
    GPT4 = "gpt-4"
    GPT4_TURBO = "gpt-4-turbo-preview"

# Message model
class Message(BaseModel):
    role: str = Field(..., description="Message role (system, user, assistant)")
    content: str = Field(..., description="Message content")
    
    @validator('role')
    def validate_role(cls, v):
        if v not in ['system', 'user', 'assistant']:
            raise ValueError('Invalid role')
        return v

# Advanced chat request
class AdvancedChatRequest(BaseModel):
    messages: List[Message] = Field(..., description="Conversation messages")
    model: ModelType = Field(default=ModelType.GPT35)
    temperature: float = Field(default=0.7, ge=0, le=2)
    max_tokens: Optional[int] = Field(default=None, gt=0, le=4000)
    top_p: Optional[float] = Field(default=1.0, ge=0, le=1)
    
    class Config:
        schema_extra = {
            "example": {
                "messages": [
                    {"role": "user", "content": "Hello!"}
                ],
                "model": "gpt-3.5-turbo",
                "temperature": 0.7
            }
        }

# Detailed response
class AdvancedChatResponse(BaseModel):
    response: str
    model: str
    usage: dict = Field(..., description="Token usage")
    finish_reason: str
    latency_ms: float

print("📝 Pydantic Models")
print("\nAdvanced Request Schema:")
print(AdvancedChatRequest.schema_json(indent=2))

## ⚡ Async LLM Endpoint

In [ ]:
print("""
# Async endpoint for better performance

from fastapi import FastAPI
from openai import AsyncOpenAI
import asyncio

app = FastAPI()
async_client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))

@app.post("/api/chat")
async def chat(request: ChatRequest):
    \"\"\"Async chat endpoint.\"\"\" 
    
    # Non-blocking LLM call
    response = await async_client.chat.completions.create(
        model=request.model,
        messages=[{"role": "user", "content": request.message}],
        temperature=request.temperature
    )
    
    return {
        "response": response.choices[0].message.content,
        "tokens": response.usage.total_tokens
    }

@app.post("/api/batch")
async def batch_chat(requests: List[ChatRequest]):
    \"\"\"Process multiple requests in parallel.\"\"\" 
    
    # Run all requests concurrently
    tasks = [chat(req) for req in requests]
    results = await asyncio.gather(*tasks)
    
    return results

💡 Benefits:
  - Non-blocking I/O
  - Handle more concurrent requests
  - Better resource utilization
  - Parallel batch processing
""")

## 🛡️ Error Handling

In [ ]:
print("""
from fastapi import HTTPException, status
from fastapi.responses import JSONResponse

# Custom exception handler
@app.exception_handler(ValueError)
async def value_error_handler(request, exc):
    return JSONResponse(
        status_code=400,
        content={"error": str(exc)}
    )

@app.post("/api/chat")
async def chat(request: ChatRequest):
    try:
        # Validate
        if len(request.message) > 1000:
            raise HTTPException(
                status_code=400,
                detail="Message too long (max 1000 chars)"
            )
        
        # Call LLM
        response = await async_client.chat.completions.create(...)
        
        return {"response": response.choices[0].message.content}
    
    except openai.RateLimitError:
        raise HTTPException(
            status_code=429,
            detail="Rate limit exceeded. Please try again later."
        )
    
    except openai.APIError as e:
        raise HTTPException(
            status_code=502,
            detail=f"OpenAI API error: {str(e)}"
        )
    
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail="Internal server error"
        )

HTTP Status Codes:
  200: Success
  400: Bad Request (invalid input)
  401: Unauthorized
  429: Rate Limit
  500: Internal Error
  502: Backend Error (OpenAI down)
""")

## ✅ Summary

### FastAPI Basics:

**1. Define Models**
```python
from pydantic import BaseModel

class Request(BaseModel):
    message: str
```

**2. Create Endpoint**
```python
@app.post("/api/chat")
async def chat(request: Request):
    return {"response": "..."}
```

**3. Run Server**
```bash
uvicorn main:app --reload
```

**4. Access Docs**
```
http://localhost:8000/docs  # Swagger UI
http://localhost:8000/redoc # ReDoc
```

### Key Features:

**Auto Validation:**
```python
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=1000)
    # FastAPI validates automatically!
```

**Type Hints:**
```python
@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest) -> ChatResponse:
    # Full type safety
```

**Async Support:**
```python
async def chat(request: ChatRequest):
    response = await async_client.chat.completions.create(...)
    # Non-blocking!
```

### Best Practices:

**1. Use Pydantic Models**
- Define request/response schemas
- Add validation
- Include examples

**2. Make It Async**
- Use AsyncOpenAI
- Async endpoints
- Better performance

**3. Handle Errors**
- Specific error codes
- Clear error messages
- Don't expose internals

**4. Add Documentation**
- Endpoint descriptions
- Parameter docs
- Example requests

### Next: `08_production_apis/02_streaming.ipynb`